In [1]:
import os
import pandas as pd
import re
from datetime import datetime
import gc
import numpy as np


In [ ]:
# Path to the current folder containing
folder_path = os.getcwd() 

# Get a list of all CSV files in the folder
file_list = sorted([f for f in os.listdir(folder_path) if f.endswith('.csv')], key=str.casefold)
print("\n".join(file_list))

In [3]:
# Initialize the variables
current_file_index = -1
release_date = None
month = None
year = None
year_month = None

Functions:

In [4]:
import os
import pandas as pd

def load_next_csv(file_list, folder_path, current_file_index):
    """
    Loads the next CSV file from the file list into a DataFrame.
    
    Parameters:
    file_list (list): List of CSV filenames to load.
    folder_path (str): Path to the folder containing the CSV files.
    current_file_index (int): Index of the current file to load.
    
    Returns:
    tuple: DataFrame loaded from the next CSV file, updated file index
    """
    # Check if there are more files to process
    if current_file_index >= len(file_list) - 1:
        print("No more CSV files left to load.")
        return None, current_file_index
    
    # Increment the file index to load the next file
    current_file_index += 1
    
    # Get the next file in the list
    current_file = file_list[current_file_index]
    file_path = os.path.join(folder_path, current_file)
    print(f"Opening file: {current_file}")
    
    # Load the CSV file into a DataFrame without headers for initial inspection
    df = pd.read_csv(file_path, header=None)
    
    return df, current_file_index


In [5]:
def drop_nan(df, column_name):
    """
    Cleans the DataFrame by removing rows where the specified column has NaN,
    removing columns with NaN in the header, and renaming unnamed columns.
    
    Parameters:
    df (pd.DataFrame): The DataFrame to clean.
    column_name (str): The name of the column to check for NaN values in rows.
    
    Returns:
    pd.DataFrame: The cleaned DataFrame.
    """
    # Track rows with NaN in the specified column
    rows_to_delete = df[df[column_name].isna()].index.tolist()

    # Drop rows with NaN in the specified column
    df = df.dropna(subset=[column_name])

    # Track columns with NaN in the header
    columns_to_delete = [col for col in df.columns if pd.isna(col)]
    
    # Replace NaN headers with placeholders (e.g., "Unnamed")
    df.columns = [col if pd.notna(col) else f"Unnamed_{i}" for i, col in enumerate(df.columns)]
    
    # Optionally drop columns that were NaN if not needed
    df = df.loc[:, ~df.columns.str.startswith("Unnamed")]

    # Print deleted rows and columns
    print("Deleted rows (indices):", rows_to_delete)
    print("Deleted columns:", columns_to_delete)

    return df


In [6]:
def clean_dataframe(df, crisis_id_col):
    """
    Cleans the DataFrame by dropping rows where the Crisis ID column has NaN,
    resetting the index, and setting row 0 as the header.
    
    Parameters:
    df (pd.DataFrame): The DataFrame to clean.
    crisis_id_col (str): The name of the Crisis ID column to check for NaN values.
    
    Returns:
    pd.DataFrame: The cleaned DataFrame with updated headers.
    """
    # List to collect rows to drop
    rows_to_drop = []
    cols_to_drop = []
    
    # Identify rows where the Crisis ID column has NaN
    for row in range(len(df)):
        if pd.isna(df.iloc[row, crisis_id_col]):
            rows_to_drop.append(row)
    
    # Drop rows with NaN in the Crisis ID column
    print(f"Dropping NaN rows: {rows_to_drop}")
    df = df.drop(index=rows_to_drop)

    # Reset the index
    df = df.reset_index(drop=True)

    # Identify columns where header is NaN
    for col in range(0, len(df.columns)):
        if pd.isna(df.iloc[0, col]):
            cols_to_drop.append(df.columns[col])

    # Drop columns with NaN in the header
    print(f"Dropping NaN columns: {cols_to_drop}")
    df = df.drop(columns=cols_to_drop)

    # Make row 0 the header
    df.columns = df.iloc[0]
    df = df.drop(df.index[0])

    # Rename columns for consistency
    column_mappings = {
        'Crisis Id': 'Crisis Id',
        'Crisisid': 'Crisis Id',
        'CrisisID': 'Crisis Id',
        'ISO3 code': 'ISO3',
        'Iso3 Code': 'ISO3',
        'Iso3': 'ISO3',
        'Iso 3': 'ISO3',
        # Add any other variations that need standardization
    }
    df = df.rename(columns=column_mappings)

    # Standardize target columns to titlecase strings and strip whitespace
    target_columns = ['Crisis', 'Drivers', 'Crisis Id', 'Country', 'ISO3']
    for col in target_columns:
        if col in df.columns:
            df[col] = df[col].astype(str).str.title().str.strip()

    df.columns = df.columns.str.title()

    return df


In [7]:
def get_dataframe_stats(df):
    """
    Prints the number of rows and columns in the DataFrame.
    
    Parameters:
    df (pd.DataFrame): The DataFrame to analyze.
    """
    num_rows, num_columns = df.shape
    return f"{num_rows} rows x {num_columns} columns"

In [8]:
#####-MERGE-#####


def merge_df(df_1, df_2, on_col, merge_type='outer'):
    # Add suffixes to all columns except the key column
    df_1 = df_1.rename(columns={col: f"{col}[df1]" for col in df_1.columns if col != on_col})
    df_2 = df_2.rename(columns={col: f"{col}[df2]" for col in df_2.columns if col != on_col})
    
    # Merge the DataFrames
    merged_df = pd.merge(df_1, df_2, on=on_col, how=merge_type)

    # Remove suffixes by renaming columns
    merged_df.columns = [col.replace('[df1]', '').replace('[df2]', '') for col in merged_df.columns]
    
    # Identify duplicated columns (keeping only the first occurrence)
    duplicate_columns = [col for col in merged_df.columns if merged_df.columns.tolist().count(col) > 1]

    # Remove duplicate columns, keeping only the first occurrence
    merged_df = merged_df.loc[:, ~merged_df.columns.duplicated()]

    # Identify rows where all values are duplicates
    duplicate_rows = merged_df.duplicated(keep=False)  # Marks all rows that are full duplicates

    # Get the indices of duplicate rows
    duplicate_indices = merged_df[duplicate_rows].index.tolist()

    # Count the number of duplicate rows
    num_deleted_rows = len(duplicate_indices)

    # Drop these duplicate rows from the DataFrame
    merged_df_cleaned = merged_df[~duplicate_rows].reset_index(drop=True)

    # Print the number of deleted rows and the row indices
    print(f"Number of rows deleted: {num_deleted_rows}")
    print("Indices of deleted duplicate rows:", duplicate_indices)
    print("DataFrame after removing fully duplicate rows:")
    print(merged_df_cleaned)

    return merged_df_cleaned



About.csv

In [ ]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")
df

In [10]:
# Function to check if a string is a date in different formats
def is_date(string):
    formats = ["%d/%m/%Y", "%Y-%m-%d %H:%M:%S"]
    for fmt in formats:
        try:
            return datetime.strptime(string, fmt).strftime("%d/%m/%Y")
        except ValueError:
            continue
    return False

In [ ]:
current_file = file_list[current_file_index]
print(f"Current file: {current_file}")
if current_file.endswith("About.csv"):
    # Identify the first column by its index position
    first_column = df.columns[0]
    
    # Search for the word 'release' in the first column
    for i, row in df.iterrows():
        if str(row[first_column]).strip().lower() == "release:":
            # Check if the next row exists and if it contains a date
            next_row_value = df.at[i + 1, first_column] if i + 1 < len(df) else None
            formatted_date = is_date(str(next_row_value))
            if formatted_date:
                release_date = formatted_date
                day, month, year = release_date.split("/")  # Split the date into parts
                month = int(month)
                year = int(year)
                year_month = f"{year}_{month:02d}"
            break

In [ ]:
# Display the extracted release date, month, year, and year_month
if release_date:
    print(f"Release date for {current_file}: {release_date}")
    print(f"Month: {month}")
    print(f"Year: {year}")
    print(f"Year-Month: {year_month}")
else:
    print("\n" + f"No release date found in {current_file}.")

In [13]:
pd.set_option('display.max_columns', None)


df1. Complexity_of_the_crisis.csv

In [ ]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")
df

In [ ]:
# Case-Specific
current_file = file_list[current_file_index]
if current_file.endswith("Complexity_of_the_crisis.csv"):
    if df.iloc[1, 5] == "Empowerment" and pd.isna(df.iloc[4, 5]):
        # Copy the content from row 1 (columns 5-39) down into row 4 (columns 5-39)
        df.iloc[4, 5:df.shape[1]] = df.iloc[1, 5:df.shape[1]]
    # Delete empty rows
    df = clean_dataframe(df, crisis_id_col = 2)
    df1 = df
else: print("Wrong file")

In [ ]:
# Display the updated DataFrame
df1

df2. Conditions_of_people_affected.csv

In [ ]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")
df

In [ ]:
# Case-Specific
current_file = file_list[current_file_index]
if current_file.endswith("Conditions_of_people_affected.csv"):
    if df.iloc[1, 5] == "# of people in none/minimal conditions - Level 1" and pd.isna(df.iloc[4, 5]):
        # Copy the content from row 1 (columns 5-39) down into row 4 (columns 5-39)
        df.iloc[4, 5:df.shape[1]] = df.iloc[1, 5:df.shape[1]]
    # Delete empty rows
    df = clean_dataframe(df, crisis_id_col = 2)
    df2 = df
else: print("Wrong file")

In [ ]:
# Display the updated DataFrame
df2

In [ ]:
#####-MERGE-#####

merged_df = merge_df(df1, df2, on_col="Crisis Id")
merged_df

df3. Core_Indicators.csv

In [ ]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")
df


In [ ]:
def find_column_name(df, row, col):
    # Start one row up from the "Helper" cell
    current_row = row - 1
    current_col = col  # Starting column

    # Iterate to the right until a non-empty cell is found
    while current_col < df.shape[1]:  # Ensure we don't go beyond the last column
        if pd.notna(df.iloc[current_row, current_col]):
            return df.iloc[current_row, current_col]
        current_col += 1  # Move to the next column

    # If no column name is found, return a placeholder or None
    return None

# Case-Specific
current_file = file_list[current_file_index]
if current_file.endswith("Core_Indicators.csv"):
    for col in range(5, len(df.columns)):
        if pd.notna(df.loc[1, col]) and re.match(r"^Helper.*", str(df.loc[1, col])):
        # Try to find the column name by searching upwards
            col_name = find_column_name(df, 1, col)
        df.loc[0, col] =f"{col_name} [{str(df.loc[1, col])}]"
    df = clean_dataframe(df, crisis_id_col = 3)
    
    df3 = df
else: print("Wrong file")

In [ ]:
df3

In [ ]:
#####-MERGE-#####

merged_df = merge_df(merged_df, df3, on_col="Crisis Id")
merged_df

df4. Country_Indicator_Data.csv

In [ ]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")
df

In [ ]:
# Case-Specific
current_file = file_list[current_file_index]
if current_file.endswith("Country_Indicator_Data.csv"):
    cols_to_drop = []
    for col in range(2, len(df.columns)):
        if pd.isna(df.iloc[0, col]):
            cols_to_drop.append(df.columns[col])
            df[col] = df[col].astype(object)  # Convert the entire column to object type
        df.iloc[0, col] =f"{df.iloc[0, col]} [{str(df.iloc[1, col])}] [{df.iloc[2, col]}]"

    print(f"Columns to drop: {cols_to_drop}")
    df = df.drop(columns=cols_to_drop)

    df = clean_dataframe(df, crisis_id_col = 1)
    df4 = df
else: print("Wrong file")

In [ ]:
df4

In [ ]:
#####-MERGE-#####
merged_df = merge_df(merged_df, df4, on_col="Country")
merged_df

df5. Crisis_Indicator_Data.csv

In [ ]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")
df

In [ ]:
# Case-Specific
current_file = file_list[current_file_index]
if current_file.endswith("Crisis_Indicator_Data.csv"):
    for col in range(5, len(df.columns)):
        if pd.notna(df.loc[1, col]):
            df.loc[0, col] =f"{df.loc[0, col]} [{df.loc[1, col]}]" 
    df = clean_dataframe(df, crisis_id_col = 2)
    df5 = df
else: print("Wrong file")

In [ ]:
df5

In [ ]:
merged_df = merge_df(merged_df, df5, on_col="Crisis Id")
merged_df

df6. Crisis_info.csv

In [ ]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")
df

In [ ]:
# Case-Specific
current_file = file_list[current_file_index]
if current_file.endswith("Crisis_info.csv"):
    df = clean_dataframe(df, crisis_id_col = 0)
    df6 = df
else: print("Wrong file")

In [ ]:
df6

In [ ]:
#####-MERGE-#####

merged_df = merge_df(merged_df, df6, on_col="Crisis Id")
merged_df

df7. Data_Reliability.csv

In [ ]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")
df

In [ ]:
# Case-Specific
current_file = file_list[current_file_index]
if current_file.endswith("Data_Reliability.csv"):
    df = clean_dataframe(df, crisis_id_col = 0)
    df7 = df
else: print("Wrong file")

In [ ]:
df7['Crisis Id'] = [id.upper() for id in df7['Crisis Id']]
df7

In [ ]:
#####-MERGE-#####

merged_df = merge_df(merged_df, df7, on_col="Crisis Id")
merged_df

df8. Impact_of_the_crisis.csv

In [ ]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")
df

In [ ]:
# Case-Specific

current_file = file_list[current_file_index]
if current_file.endswith("Impact_of_the_crisis.csv"):
    if df.iloc[1, 5] == "Area affected - absolute" and pd.isna(df.iloc[4, 5]):
        # Copy the content from row 1 (columns 5-...) down into row 4 (columns 5-...)
        df.iloc[4, 5:df.shape[1]] = df.iloc[1, 5:df.shape[1]]
    df = clean_dataframe(df, crisis_id_col = 0)
    df8 = df
else: print("Wrong file")

In [ ]:
df8

In [ ]:
#####-MERGE-#####

merged_df = merge_df(merged_df, df8, on_col="Crisis Id")
merged_df

Imputed_and_missing_data_hidden.csv -- No data

In [ ]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")


Indicator_Date_hidden.csv -- No data

In [ ]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")


Indicator_Date_hidden2.csv -- No data

In [ ]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")


Indicator_Metadata.csv -- No data, just description of features

In [ ]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")


In [ ]:
df = clean_dataframe(df, crisis_id_col = 0)
df.head()

df9. all_crises.csv

In [ ]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")
df

In [ ]:
# Case-Specific

current_file = file_list[current_file_index]
if current_file.endswith("all_crises.csv"):
    df = clean_dataframe(df, crisis_id_col = 1)
    df9 = df
else: print("Wrong file")

In [ ]:
df9

In [ ]:
#####-MERGE-#####

merged_df = merge_df(merged_df, df9, on_col="Crisis Id")
merged_df

df10. country.csv

In [ ]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")
df

In [ ]:
# Case-Specific

current_file = file_list[current_file_index]
if current_file.endswith("country.csv"):
    df = clean_dataframe(df, crisis_id_col = 1)
    df10 = df
else: print("Wrong file")

In [ ]:
df10

In [ ]:
#####-MERGE-#####

merged_df = merge_df(merged_df, df10, on_col="Crisis Id")
merged_df

hidden.csv -- Was hidden. No Data to be used

In [ ]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")
df

df11. Lists.csv

In [ ]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")
df

In [ ]:
# Case-Specific

current_file = file_list[current_file_index]
if current_file.endswith("Lists.csv"):
    df = clean_dataframe(df, crisis_id_col = 2)
    df11 = df
else: print("Wrong file")

In [ ]:
df11

In [ ]:
#####-MERGE-#####

merged_df = merge_df(merged_df, df11, on_col="Crisis Id")
merged_df

df12. Log.csv

In [ ]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")
df

In [ ]:
# Case-Specific

current_file = file_list[current_file_index]
if current_file.endswith("Log.csv"):
    df = clean_dataframe(df, crisis_id_col = 1)
    df12 = df
else: 
    print("Wrong file")
    current_file_index -= 1

In [ ]:
df12

In [ ]:
#####-MERGE-#####

# merged_df = merge_df(merged_df, df12, on_col="Crisis Id")
merged_df

df13. Regional_Crises.csv

In [ ]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")
df

In [ ]:
# Case-Specific

current_file = file_list[current_file_index]
if current_file.endswith("Regional_Crises.csv"):
    for col in range(5, len(df.columns)):
        if pd.notna(df.loc[1, col]):
            df.loc[0, col] =f"{df.loc[0, col]} [{df.loc[1, col]}]" 
    df = clean_dataframe(df, crisis_id_col = 1)
    df13 = df
else: print("Wrong file")

In [ ]:
df13

In [ ]:
#####-MERGE-#####

merged_df = merge_df(merged_df, df13, on_col="Crisis Id")
merged_df

df14. Reliability.csv

In [ ]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")
df

In [ ]:
# Case-Specific

current_file = file_list[current_file_index]
if current_file.endswith("Reliability.csv"):
    df = clean_dataframe(df, crisis_id_col = 0)
    df14 = df
else: print("Wrong file")

In [ ]:
df14

In [ ]:
#####-MERGE-#####

merged_df = merge_df(merged_df, df14, on_col="Crisis Id")
merged_df

df15. Reliability_updated.csv.csv

In [ ]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")
df

In [ ]:
# Case-Specific

current_file = file_list[current_file_index]
if current_file.endswith("Reliability_updated.csv"):
    df = clean_dataframe(df, crisis_id_col = 0)
    df15 = df
else: print("Wrong file")


In [ ]:
df15

In [ ]:
#####-MERGE-#####

merged_df = merge_df(merged_df, df15, on_col="Crisis Id")
merged_df

df16. Trends.csv

In [ ]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")
df

In [ ]:
# Case-Specific

current_file = file_list[current_file_index]
if current_file.endswith("Trends.csv"):
    df = clean_dataframe(df, crisis_id_col = 2)
    df16 = df
else: print("Wrong file")


In [ ]:
df16

In [ ]:
#####-MERGE-#####

# merged_df = merge_df(merged_df, df16, on_col="Crisis Id")
merged_df

In [83]:
# Remove rows where there are NaNs in the Crisis Id column and headers with NaNs or Unnamed
# merged_df =  drop_nan(merged_df, column_name="Crisis Id")

In [ ]:
# Add the year_month column as the first column
merged_df.insert(0, 'YYYY_MM', year_month)

# Reorder columns to have Crisis ID as the second column
columns = ['YYYY_MM', 'Crisis Id'] + [col for col in merged_df.columns if col not in ['YYYY_MM', 'Crisis Id']]
merged_df = merged_df[columns]

merged_df


In [89]:
merged_df.to_csv(f"{year_month}_merged.csv", index=False)

In [ ]:
print(year_month)

In [ ]:
final_df = pd.read_csv(f"{year_month}_merged.csv")
final_df.shape

In [ ]:
# print(final_df.name)